In [1]:
# conda vis2
from pathlib import Path
import sys
import numpy as np
import plotly.graph_objects as go
import re

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.ExperimentSet import ExperimentSet

In [3]:
def get_unique_path(path):
    path = Path(path)
    if not path.exists():
        return path

    match = re.search(r'(\d+)$', path.stem)
    if match:
        base = path.stem[:match.start()]
        num = int(match.group(1)) + 1
    else:
        base = path.stem
        num = 1

    new_path = path.with_name(f"{base}{num}{path.suffix}")
    # keep incrementing until we find one that doesn't exist
    while new_path.exists():
        num += 1
        new_path = path.with_name(f"{base}{num}{path.suffix}")

    return new_path

In [4]:
es_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V11'
exset = ExperimentSet(es_dir)
exset.set_retrieval_list()
exset.load_retrievals(iteration=-1, extinction_th=[0.3, np.inf])

In [8]:
isomin=1
isomax=40
opacity=0.1
surface_count=30
cmin=0
cmax=40

cloud = exset.get_retrieval('default').solvers_retrieval[0.67].medium['cloud']
cond = cloud.extinction.data >= 0.3

X, Y, Z = np.meshgrid(cloud.x.data, cloud.y.data, cloud.z.data, indexing='ij')
data = go.Volume(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=cloud.extinction.data.flatten(),
    isomin=isomin,
    isomax=isomax,
    opacity=opacity,
    surface_count=surface_count,
    showscale=True,
    cmin=cmin,
    cmax=cmax,
    # colorscale='YlOrBr',
    colorscale='Blues',
    colorbar=dict(
        title='',
        titleside='right',
        tickmode='array',
        tickvals=[cmin, 0.5*(cmin+cmax), cmax],
        ticktext=['Low', 'Medium', 'High'],
        len=0.4,
        thickness=20,
        tickfont=dict(size=20)
    )
)
layout = go.Layout(autosize=False, width=750, height=650)
fig = go.Figure(data=data, layout=layout)
pad = {'b':0, 'l':0, 'r':0, 't':0, 'pad':0}
font = {'size':16}
# scene = dict(
#     xaxis=dict(title='x (km)', tickvals=[0, 4.4, 8.8, 13.2, 17.6], range=[-1, 18.6]),
#     yaxis=dict(title='y (km)', tickvals=[0, 4.4, 8.8, 13.2, 17.6], range=[-1, 18.6]),
#     zaxis=dict(title='z (km)', tickvals=[0, 1.4], range=[-0.1, 1.5]),
# )
scene = dict(
    xaxis=dict(title='', tickvals=[], range=[-1, 18.6]),
    yaxis=dict(title='', tickvals=[], range=[-1, 18.6]),
    zaxis=dict(title='', tickvals=[], range=[-0.1, 1.5]),
    aspectmode='manual',
    aspectratio=dict(x=1,y=1,z=0.5)
    # aspectmode='data'
)

theta = np.radians(68)
phi = -0.33*np.pi
radius = 2.0
h = radius * np.sin(theta)

camera = dict(
    up=dict(x=0, y=0, z=1.0),
    center=dict(x=0, y=0, z=0),
    eye=dict(x=h*np.cos(phi), y=h*np.sin(phi), z=radius*np.cos(theta))
)
fig.update_layout(
    scene_camera=camera,
    margin=pad,
    font=font,
    scene=scene,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(1,1,1,1)'
)

outfile = PROJECT_ROOT / 'notebooks' / 'AMS_Madison_2026_figures' / 'extinction3D_1.png'
outfile = get_unique_path(outfile)
fig.write_image(str(outfile), scale=4)
